# Focal-plane field atlas: Gaussian, LG, cylindrical-vector and full-Poincaré beams
### Richards–Wolf vectorial diffraction • method: **five Bessel integrals** • runtime: **JAX (GPU required by default)**

A self-contained research and teaching notebook for the ideal aplanatic objective. Every example calculates the **complex** focal-plane components $E_x(x,y,0),E_y(x,y,0),E_z(x,y,0)$, plots six intensity/phase maps, and constructs an **8×8 grid of true 3D polarization ellipses** in PyVista over the total intensity pattern.

**Organization**  
1. Gaussian ($\ell=0$): H/X, V/Y, diagonal D, antidiagonal A, R and L circular.  
2. LG$_0^{+1}$ and LG$_0^{-1}$: six input Jones states **for each charge**.  
3. Four cylindrical vector beams: radial, azimuthal, hyperbolic-x, hyperbolic-y.  
4. Full-Poincaré beam: star, monstar, lemon (thesis Eq. 10 convention).

**Source:** M. F. Ferrer-García, PhD thesis, Chapter 2 and Appendix A; thesis-associated 2020 Optics Express article, cylindrical-vector Eq. (9), full-Poincaré Eq. (10). The four chosen CV examples are an illustrative basis of the cylindrical vector family, not four exclusive classes.

**Conventions:** harmonic time dependence $\mathrm{Re}[\mathbf E e^{-i\omega t}]$, propagation $e^{+ikz}$, pupil ray map $r_p=f\sin\theta$, input $E_z^{\mathrm{in}}=0$. $\lambda_0$ denotes vacuum wavelength. R/L use the explicitly declared Jones vectors below; nomenclature differs among optics communities.

**Performance:** complex128; JAX variants require an accelerator by default and transfer completed results to the CPU for plotting. All PyVista plots are produced by running the notebook; screenshots require PyVista + an off-screen rendering backend. The focus is on physics and reproducibility, not precomputed example images.

## 0. Installation and numerical parameters

CPU: `%pip install numpy scipy matplotlib pyvista jupyterlab`. GPU/JAX: install a **CUDA-compatible JAX build matching your CUDA environment** following the official JAX instructions, plus `pyvista` and `matplotlib`. PyVista may need a functional OpenGL/Xvfb setup in headless Colab; the notebook will report an actionable error if rendering is unavailable.

We choose $\mathrm{NA}=0.85<n=1$, wavelength units $\lambda_0=1$, and field-of-view $\sim 8\lambda_0$. These are examples; vary resolution, pupil cutoff and window independently in validation. A finite FFT grid produces a periodic approximation, while the radial integral solver interpolates sampled radial integrals. Neither is exact at finite resolution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv
from scipy.integrate import simpson
from functools import lru_cache
from IPython.display import display, Image
NA=0.85; n_medium=1.0; lambda0=1.0; f=1.0
k=2*np.pi*n_medium/lambda0; k0=2*np.pi/lambda0
alpha=np.arcsin(NA/n_medium)
N=121 if "integral_jax".startswith("integral") else 192
dx=0.045 if "integral_jax".startswith("integral") else 0.045
x=(np.arange(N)-(N-1)/2)*dx
X,Y=np.meshgrid(x,x,indexing="xy")
RHO=np.hypot(X,Y); PHI=np.arctan2(Y,X)
# Keep these independent so backend dispatch does not silently change the optical setup.
print(f"method=integral_jax, NA={NA}, lambda0={lambda0}, N={N}, dx={dx}, extent={x[0]:.2f}...{x[-1]:.2f}")
if NA>=n_medium: raise ValueError("Require NA<n for this example.")

## 0A. Explicit Jones polarization conventions and beam decomposition

This notebook uses `R=(xhat - i yhat)/sqrt(2)`, `L=(xhat + i yhat)/sqrt(2)` with $\exp(-i\omega t)$. They describe opposite rotation senses, but authors may reverse the labels depending on their viewing direction. Define $D=(H+V)/\sqrt2$ and $A=(H-V)/\sqrt2$.

Every input is a **linear combination of scalar LG-like pupil modes** $h_{\ell}(q)e^{i\ell\phi}$ multiplying a complex Jones vector. Here $q=\sin\theta/\sin\alpha$ and $h_{\ell}(q)=q^{|\ell|}e^{-(q/\mathrm{fill})^2}$, with $\mathrm{fill}=1$.

**Cylindrical construction (thesis Eq. 9):** equal-weight opposite-helicity modes with charges $\pm1$. Reversing charge assignment gives the two hyperbolic counterparts. **Full-Poincaré construction (thesis Eq. 10):** $\cos\alpha_p(\cos\beta\,LG_{+1}-\sin\beta\,LG_{-1})\mathbf e_R+e^{i\delta}\sin\alpha_p\,LG_{0}\mathbf e_L$, with $\alpha_p=\pi/4$ and $\delta=0$. The labels star/monstar/lemon follow the thesis's $\beta=0,\pi/4,\pi/2$ parameterization. A definitive post-focusing classification additionally requires computing polarization-line indices, since high-NA focusing can modify the local texture.

In [ ]:
SQ2=np.sqrt(2.0)
JONES={
 "X / H":np.array([1,0],complex),
 "Y / V":np.array([0,1],complex),
 "D (+45°)":np.array([1,1],complex)/SQ2,
 "A (-45°)":np.array([1,-1],complex)/SQ2,
 "R":np.array([1,-1j],complex)/SQ2,
 "L":np.array([1,1j],complex)/SQ2,
}
def one_mode(ell,jones,weight=1):
    return (int(ell),complex(jones[0])*weight,complex(jones[1])*weight)
def gaussian(jones): return [one_mode(0,jones)]
def lg(ell,jones): return [one_mode(ell,jones)]
def cylindrical(kind):
    R,L=JONES["R"],JONES["L"]
    if kind=="Radial": return [one_mode(+1,R,1/SQ2),one_mode(-1,L,1/SQ2)]
    if kind=="Azimuthal": return [one_mode(+1,R,1/SQ2),one_mode(-1,L,-1/SQ2)]
    if kind=="Hyperbolic X": return [one_mode(-1,R,1/SQ2),one_mode(+1,L,1/SQ2)]
    if kind=="Hyperbolic Y": return [one_mode(-1,R,1/SQ2),one_mode(+1,L,-1/SQ2)]
    raise ValueError(kind)
def poincare(beta):
    R,L=JONES["R"],JONES["L"]
    return [one_mode(+1,R,np.cos(beta)/SQ2),one_mode(-1,R,-np.sin(beta)/SQ2),
            one_mode(0,L,1/SQ2)]
def pupil_mode(q,phi,ell):
    return (q**abs(ell))*np.exp(-q*q)*np.exp(1j*ell*phi)
print("Input modes initialized; each case is a list of (charge, Jones-x, Jones-y).")

## 0B. Five-integral numerical backend

The analytic azimuthal integration produces Bessel orders $\ell$, $\ell\pm1$, and $\ell\pm2$. The **longitudinal** orders $\ell\pm1$ include an additional $\sin\theta$ in their $\theta$ integrands, as required by the third row of the lens matrix. The code below uses exactly this corrected thesis convention.

The radial integrals depend only on $\rho$, so they are evaluated on a dense radial grid **once per charge**, then interpolated to the Cartesian plotting grid. This is the thesis-based approach and avoids evaluating Bessel functions at every pixel. Approximation errors come from angular quadrature and radial interpolation.

**GPU variant:** JAX evaluates integer-order Bessel functions via `jax.scipy.special.bessel_jn` and computes all radial integrals on the accelerator. The result is transferred to NumPy for assembly/visualization. The JAX GPU device is checked explicitly, and no CPU fallback is presented as GPU acceleration.


In [ ]:
import jax
jax.config.update("jax_enable_x64",True)
import jax.numpy as jnp
from jax.scipy.special import bessel_jn
import warnings
GPU_DEVICES=[d for d in jax.devices() if d.platform=="gpu"]
REQUIRE_GPU=True
if REQUIRE_GPU and not GPU_DEVICES:
    raise RuntimeError("No JAX GPU detected. In Colab choose an NVIDIA GPU runtime and install the CUDA JAX wheel; for CPU debugging only, set REQUIRE_GPU=False.")
print("JAX computation device:",jax.devices()[0])
# Derive J_n for any n in [-3,3] from nonnegative orders.
def int_bessel(order,arg):
    v=abs(int(order))
    ans=bessel_jn(arg,v=v)[v]
    return ans*((-1)**v if order<0 else 1)


In [ ]:
def jax_trap(values,axis_vals):
    return np.asarray(jnp.trapezoid(values,x=jnp.asarray(axis_vals),axis=1))
radial_nodes=np.linspace(0,float(np.max(RHO)),251)
theta=np.linspace(0,alpha,401)
def make_basis(ell):
    ell=int(ell)
    T=jnp.asarray(theta)[None,:]; U=jnp.asarray(radial_nodes)[:,None]
    q=np.sin(T)/np.sin(alpha)
    g=np.sqrt(np.cos(T))*np.sin(T)*(q**abs(ell))*np.exp(-q*q)
    arg=k*U*np.sin(T)
    ct=np.cos(T); st=np.sin(T)
    b1=jax_trap(g*(1+ct)*int_bessel(ell,arg),theta)
    b2=jax_trap(g*(1-ct)*BESSEL(ell+2,arg),theta)
    b3=jax_trap(g*(1-ct)*BESSEL(ell-2,arg),theta)
    b4=jax_trap(g*st*BESSEL(ell+1,arg),theta)
    b5=jax_trap(g*st*BESSEL(ell-1,arg),theta)
    fac=-np.pi*(1j**(ell+1))*f/(lambda0/n_medium)
    flong=np.pi*(1j**ell)*f/(lambda0/n_medium)
    def interp(a):
        a=np.asarray(a); r=RHO.ravel()
        return (np.interp(r,radial_nodes,a.real)+1j*np.interp(r,radial_nodes,a.imag)).reshape(RHO.shape)
    I1=fac*interp(b1)*np.exp(1j*ell*PHI)
    I2=fac*interp(b2)*np.exp(1j*(ell+2)*PHI)
    I3=fac*interp(b3)*np.exp(1j*(ell-2)*PHI)
    I4=flong*interp(b4)*np.exp(1j*(ell+1)*PHI)
    I5=flong*interp(b5)*np.exp(1j*(ell-1)*PHI)
    # Basis output [input x/y, focal Ex/Ey/Ez, y, x].
    F_x=np.stack([(2*I1+I2+I3)/2,(I2-I3)/(2j),I5-I4])
    F_y=np.stack([(I2-I3)/(2j),(2*I1-I2-I3)/2,1j*(I4+I5)])
    return np.stack([F_x,F_y])
@lru_cache(None)
def cached_basis(ell): return make_basis(ell)
def calculate(terms):
    E=np.zeros((3,N,N),complex)
    for ell,ax,ay in terms:
        B=cached_basis(ell)
        E+=ax*B[0]+ay*B[1]
    return E


## 0C. Physics checks and visualization tools

The polarization ellipse at each point is the trajectory
$\mathbf r_\mathrm{ellipse}(t)=\mathrm{Re}[\mathbf E(x,y,0)e^{-it}]=\mathrm{Re}\mathbf E\cos t+\mathrm{Im}\mathbf E\sin t$.
This is a **true 3D ellipse** because the focused field has a nonzero $E_z$.

The sample coordinates form an **8×8** grid. Ellipse sizes are proportional to their local field amplitude, with one *global display scale* per panel; weak field sites are skipped so noise phases do not produce misleading polarization arrows. The background plane is colored by normalized total intensity. Ellipse hue encodes the sign of $\mathrm{Im}(E_xE_y^*)$ **only as a transverse handedness proxy**, not a complete classification of three-dimensional handedness.

The plotted scalar phases are the arguments of the **complex Cartesian components**, not a uniquely defined phase for the total vector intensity. Low-amplitude pixels have undefined phase and are masked on each component plot. Nothing is modified in the complex field before ellipse reconstruction.

In [ ]:
def maps(E,title):
    I=np.abs(E)**2; Itotal=np.sum(I,axis=0)
    fig,axs=plt.subplots(2,3,figsize=(12.5,7.0),constrained_layout=True)
    for j,label in enumerate(("Ex","Ey","Ez")):
        axs[0,j].imshow(I[j],origin="lower",extent=[x[0],x[-1],x[0],x[-1]])
        axs[0,j].set_title(f"|{label}|²")
        ang=np.angle(E[j]); ang=np.ma.array(ang,mask=I[j]<max(1e-15,1e-5*np.max(I[j])))
        axs[1,j].imshow(ang,origin="lower",extent=[x[0],x[-1],x[0],x[-1]],
                        cmap="twilight",vmin=-np.pi,vmax=np.pi)
        axs[1,j].set_title(f"arg({label})")
    fig.suptitle(title+" | z = 0",fontsize=14)
    plt.show();plt.close(fig)
    print("Total intensity max:",float(Itotal.max()),
          "; longitudinal fraction:",float(I[2].sum()/max(Itotal.sum(),1e-30)))


In [ ]:
def pyvista_ellipses(E,title,grid_points=8,off_screen=True):
    try:
        import pyvista as pv
    except ImportError:
        print("PyVista unavailable. Install with %pip install pyvista and rerun this cell.")
        return None
    if grid_points!=8: raise ValueError("This atlas specifies an 8x8 ellipse grid.")
    intensity=np.sum(np.abs(E)**2,axis=0)
    intensity=intensity/max(float(np.max(intensity)),1e-30)
    try:
        mesh=pv.StructuredGrid(X,Y,np.zeros_like(X))
        mesh["Intensity"]=np.ascontiguousarray(intensity).ravel(order="F")
        plot=pv.Plotter(off_screen=off_screen,window_size=(1000,850))
        plot.set_background("white")
        plot.add_mesh(mesh,scalars="Intensity",cmap="viridis",opacity=0.83,
                      clim=[0,1],show_scalar_bar=True)
        indices=np.linspace(int(.08*(N-1)),int(.92*(N-1)),8).round().astype(int)
        emax=float(np.max(np.linalg.norm(E,axis=0)))
        site_pitch=(x[indices[-1]]-x[indices[0]])/7
        scale=0.36*site_pitch/max(emax,1e-30)
        count=0
        t=np.linspace(0,2*np.pi,73)
        for iy in indices:
            for ix in indices:
                v=E[:,iy,ix]
                if np.linalg.norm(v)<.035*emax: continue
                # Choose an overall sign-independent ellipse; real part vs imaginary part is time phase.
                curve=(v.real[None,:]*np.cos(t[:,None])+v.imag[None,:]*np.sin(t[:,None]))*scale
                curve[:,0]+=x[ix];curve[:,1]+=x[iy]
                # slight uniform vertical offset ensures ellipse stays visible over intensity plane
                curve[:,2]+=0.035*site_pitch
                poly=pv.Spline(curve,200)
                helicity=np.imag(v[0]*np.conj(v[1]))
                color="tomato" if helicity>1e-9*emax**2 else ("deepskyblue" if helicity<-1e-9*emax**2 else "white")
                plot.add_mesh(poly.tube(radius=.014*site_pitch),color=color,
                              smooth_shading=True,show_scalar_bar=False)
                count+=1
        plot.add_axes()
        plot.camera_position=[(7, -9, 9),(0,0,0),(0,0,1)]
        plot.camera.zoom(1.15)
        # screenshot triggers rendering in off-screen environments.
        png=plot.screenshot(return_img=True)
        display(Image(data=__import__("io").BytesIO(
            __import__("PIL.Image",fromlist=["Image"]).Image.fromarray(png).convert("RGB").tobytes()
        ).getvalue())) if False else display(__import__("PIL.Image",fromlist=["Image"]).Image.fromarray(png))
        plot.close()
        print(f"Rendered {count}/64 polarization ellipses (omitted sites have weak field).")
        return png
    except Exception as exc:
        print("PyVista off-screen rendering failed. On Linux/Colab configure Xvfb/OpenGL or use a supported trame backend.",repr(exc))
        try: plot.close()
        except Exception: pass
        return None
def show_case(title,terms,render_3d=True):
    E=calculate(terms)
    assert E.shape==(3,N,N) and np.all(np.isfinite(E))
    maps(E,title)
    if render_3d: pyvista_ellipses(E,title)
    return E


## 1. Gaussian beam, $LG_0^{0}$
Six polarization states are constructed from the **same scalar Gaussian pupil**. An x/y polarized Gaussian produces a longitudinal focal component after high-NA focusing. For circular illumination the local focal polarization texture is not generally a uniform transverse circle. The field is always computed from the *actual lens operator*, not a paraxial Jones-vector plot.

### 1.1. Gaussian — X / H
Input: $LG_0^0$ with Jones state **X / H**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_1=show_case("Gaussian X / H",gaussian(JONES["X / H"]))

### 1.2. Gaussian — Y / V
Input: $LG_0^0$ with Jones state **Y / V**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_2=show_case("Gaussian Y / V",gaussian(JONES["Y / V"]))

### 1.3. Gaussian — D (+45°)
Input: $LG_0^0$ with Jones state **D (+45°)**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_3=show_case("Gaussian D (+45°)",gaussian(JONES["D (+45°)"]))

### 1.4. Gaussian — A (-45°)
Input: $LG_0^0$ with Jones state **A (-45°)**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_4=show_case("Gaussian A (-45°)",gaussian(JONES["A (-45°)"]))

### 1.5. Gaussian — R
Input: $LG_0^0$ with Jones state **R**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_5=show_case("Gaussian R",gaussian(JONES["R"]))

### 1.6. Gaussian — L
Input: $LG_0^0$ with Jones state **L**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_6=show_case("Gaussian L",gaussian(JONES["L"]))

## 2.A. Laguerre–Gaussian beam $LG_0^{+1}$
The scalar pupil carries azimuthal phase $e^{i\ell\phi}$ with $\ell=1$. Spin-orbit coupling mixes azimuthal orders in the focal Cartesian components (the five-integral approach explicitly shows orders $\ell,\ell\pm1,\ell\pm2$). Examine component phases, not only doughnut-shaped intensity.

### LG charge 1, polarization X / H
The incident field is a single $p=0$ vortex pupil term with a uniform **X / H** Jones vector.

In [ ]:
E_lg_plus_1=show_case("LG ℓ=1: X / H",lg(1,JONES["X / H"]))

### LG charge 1, polarization Y / V
The incident field is a single $p=0$ vortex pupil term with a uniform **Y / V** Jones vector.

In [ ]:
E_lg_plus_2=show_case("LG ℓ=1: Y / V",lg(1,JONES["Y / V"]))

### LG charge 1, polarization D (+45°)
The incident field is a single $p=0$ vortex pupil term with a uniform **D (+45°)** Jones vector.

In [ ]:
E_lg_plus_3=show_case("LG ℓ=1: D (+45°)",lg(1,JONES["D (+45°)"]))

### LG charge 1, polarization A (-45°)
The incident field is a single $p=0$ vortex pupil term with a uniform **A (-45°)** Jones vector.

In [ ]:
E_lg_plus_4=show_case("LG ℓ=1: A (-45°)",lg(1,JONES["A (-45°)"]))

### LG charge 1, polarization R
The incident field is a single $p=0$ vortex pupil term with a uniform **R** Jones vector.

In [ ]:
E_lg_plus_5=show_case("LG ℓ=1: R",lg(1,JONES["R"]))

### LG charge 1, polarization L
The incident field is a single $p=0$ vortex pupil term with a uniform **L** Jones vector.

In [ ]:
E_lg_plus_6=show_case("LG ℓ=1: L",lg(1,JONES["L"]))

## 2.B. Laguerre–Gaussian beam $LG_0^{-1}$
The scalar pupil carries azimuthal phase $e^{i\ell\phi}$ with $\ell=-1$. Spin-orbit coupling mixes azimuthal orders in the focal Cartesian components (the five-integral approach explicitly shows orders $\ell,\ell\pm1,\ell\pm2$). Examine component phases, not only doughnut-shaped intensity.

### LG charge -1, polarization X / H
The incident field is a single $p=0$ vortex pupil term with a uniform **X / H** Jones vector.

In [ ]:
E_lg_minus_1=show_case("LG ℓ=-1: X / H",lg(-1,JONES["X / H"]))

### LG charge -1, polarization Y / V
The incident field is a single $p=0$ vortex pupil term with a uniform **Y / V** Jones vector.

In [ ]:
E_lg_minus_2=show_case("LG ℓ=-1: Y / V",lg(-1,JONES["Y / V"]))

### LG charge -1, polarization D (+45°)
The incident field is a single $p=0$ vortex pupil term with a uniform **D (+45°)** Jones vector.

In [ ]:
E_lg_minus_3=show_case("LG ℓ=-1: D (+45°)",lg(-1,JONES["D (+45°)"]))

### LG charge -1, polarization A (-45°)
The incident field is a single $p=0$ vortex pupil term with a uniform **A (-45°)** Jones vector.

In [ ]:
E_lg_minus_4=show_case("LG ℓ=-1: A (-45°)",lg(-1,JONES["A (-45°)"]))

### LG charge -1, polarization R
The incident field is a single $p=0$ vortex pupil term with a uniform **R** Jones vector.

In [ ]:
E_lg_minus_5=show_case("LG ℓ=-1: R",lg(-1,JONES["R"]))

### LG charge -1, polarization L
The incident field is a single $p=0$ vortex pupil term with a uniform **L** Jones vector.

In [ ]:
E_lg_minus_6=show_case("LG ℓ=-1: L",lg(-1,JONES["L"]))

## 3. Cylindrical-vector beams — four examples from the opposite-helicity LG family
Using the superposition of opposite circular states and opposite orbital charges (thesis Eq. 9), the **radial** and **azimuthal** patterns are obtained by changing the relative phase. Swapping the charge assignments yields **hyperbolic-x** and **hyperbolic-y** vector textures. The local input Jones vector is spatially varying, but each term is a conventional $LG_0^{\ell}$ with fixed Jones polarization, so both solvers can apply superposition without new physics.

### 3.1. Radial
Inspect how the **transverse** input linear-polarization texture becomes a 3D ellipse field after focusing. Radially polarized illumination is particularly useful for generating pronounced longitudinal fields.

In [ ]:
E_cv_1=show_case("CV Radial",cylindrical("Radial"))

### 3.2. Azimuthal
Inspect how the **transverse** input linear-polarization texture becomes a 3D ellipse field after focusing. Radially polarized illumination is particularly useful for generating pronounced longitudinal fields.

In [ ]:
E_cv_2=show_case("CV Azimuthal",cylindrical("Azimuthal"))

### 3.3. Hyperbolic X
Inspect how the **transverse** input linear-polarization texture becomes a 3D ellipse field after focusing. Radially polarized illumination is particularly useful for generating pronounced longitudinal fields.

In [ ]:
E_cv_3=show_case("CV Hyperbolic X",cylindrical("Hyperbolic X"))

### 3.4. Hyperbolic Y
Inspect how the **transverse** input linear-polarization texture becomes a 3D ellipse field after focusing. Radially polarized illumination is particularly useful for generating pronounced longitudinal fields.

In [ ]:
E_cv_4=show_case("CV Hyperbolic Y",cylindrical("Hyperbolic Y"))

## 4. Full-Poincaré beams — star, monstar, and lemon
The thesis's Eq. (10) combines $LG_0^{+1}\mathbf e_R$, $LG_0^{-1}\mathbf e_R$, and $LG_0^0\mathbf e_L$. We hold $\alpha_p=\pi/4$ and $\delta=0$, and change the coefficient-mixing angle $\beta$.

**Important distinction:** star / monstar / lemon here name the **incident target textures in the cited thesis convention**. To label a focused field's observed topological singularity rigorously, compute polarization ellipse major-axis winding and separatrix count near the relevant C-point. The 8×8 visualization illustrates geometry but is **not** by itself a topology classifier.

### 4.1. Star ($\beta=0$)
Plot full vectorial focused components and the intensity plus 3D ellipse grid. The ellipticity and orientation in the focal plane need not match the paraxial input texture.

In [ ]:
E_fp_star=show_case("Full-Poincaré Star",poincare(0))

### 4.2. Monstar ($\beta=\pi/4$)
Plot full vectorial focused components and the intensity plus 3D ellipse grid. The ellipticity and orientation in the focal plane need not match the paraxial input texture.

In [ ]:
E_fp_monstar=show_case("Full-Poincaré Monstar",poincare(np.pi/4))

### 4.3. Lemon ($\beta=\pi/2$)
Plot full vectorial focused components and the intensity plus 3D ellipse grid. The ellipticity and orientation in the focal plane need not match the paraxial input texture.

In [ ]:
E_fp_lemon=show_case("Full-Poincaré Lemon",poincare(np.pi/2))

## 5. Sanity checks, convergence, and follow-up exercises
- Confirm transverse input definitions and document whether handedness is observed along $+z$ or $-z$.
- For FFT: compare discrete FFT with a direct discrete plane-wave sum and independent Debye quadrature, not only a visually plausible plot.
- For Bessel integrals: increase `theta` and `radial_nodes` density independently; inspect phase convergence near vortex zeros.
- For GPU: record actual `jax.devices()`, compilation time, synchronized execution time, dtype, and CPU-reference complex-vector errors; no GPU speedup claim is made from these example plots.
- For polarization ellipses: the displayed 8×8 grid is a **sample** of a continuously varying vector field, not a complete measure of polarization singularities. A true 3D polarization ellipse is generally tilted out of the focal plane.
- Normalize color scales consistently across cases if a quantitative visual comparison of intensity is intended; absolute incident-mode power was **not** normalized here.
- This atlas is at **z=0 only**. Multi-plane and volume propagation belong in the separate solver tutorials.

**Topological note:** In the focal region the three-dimensional ellipse is determined by the complex vector components. The transverse polarization ellipse orientation used for C-point line classification is a different projected quantity; do not conflate these.